In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
from tqdm import tqdm

from april import Dataset
from april.processmining import ProcessMap
from april.fs import get_event_log_files
from april.fs import get_process_model_files

# Event Log information

A list of all event logs used in the evaluation

In [ ]:
logs = sorted([e.name for e in get_event_log_files() if e.p == 0.3 and "bpic" not in e.model and "0.3-1" in e.name])
print(logs)

In [ ]:

columns = ['name', 'base_name', 'num_cases', 'num_events', 'num_activities', 
           'num_attributes', 'attribute_keys', 'attribute_dims', 
           'min_attribute_dim', 'max_attribute_dim',
           'min_case_len', 'max_case_len', 'mean_case_len']
df = []
for log in tqdm(logs):
    d = Dataset(log)
    dim_min = d.attribute_dims[1:].astype(int).min() if d.attribute_dims[1:].size else None
    dim_max = d.attribute_dims[1:].astype(int).max() if d.attribute_dims[1:].size else None
    df.append([log, log.split('-')[0], d.num_cases, d.num_events, d.attribute_dims[0].astype(int), 
               d.num_attributes - 1, d.attribute_keys[1:], d.attribute_dims[1:].astype(int), dim_min, dim_max,
               d.case_lens.min(), d.case_lens.max(), d.case_lens.mean().round(2)])
event_logs = pd.DataFrame(df, columns=columns)

## Basis for Table 1 in the Paper

In [ ]:
event_logs

In [ ]:
print(event_logs[['name', 'num_cases', 'num_events', 'max_case_len']])

In [ ]:
event_logs[['base_name', 'num_activities', 'num_cases', 'num_events', 'min_attribute_dim', 'max_attribute_dim']].groupby('base_name').agg(['count', 'min', 'max'])

# Process Model Information

In [ ]:
maps = sorted([m for m in get_process_model_files()])
df = []
for process_map in tqdm(maps):
    model = ProcessMap.from_plg(process_map)

    num_variants = len(model.variants.cases)
    max_case_len = model.variants.max_case_len

    nodes = model.graph.number_of_nodes()
    edges = model.graph.number_of_edges()
    dens = nx.density(model.graph)
    in_degree = np.mean([d[1] for d in model.graph.in_degree()])
    out_degree = np.mean([d[1] for d in model.graph.out_degree()])

    df.append([nodes, edges, num_variants, max_case_len, dens, in_degree, out_degree])
process_models = pd.DataFrame(df, index=maps, columns=['nodes', 'edges', 'num_variants', 'max_case_len', 'density', 'in_deg', 'out_deg'])

In [ ]:
# process_models.loc[['paper', 'p2p', 'small', 'medium', 'large', 'huge', 'gigantic', 'wide', 'testing']].round(2)
process_models.loc[['paper', 'p2p', 'small', 'medium', 'large', 'huge', 'gigantic', 'wide']].round(2)
